# 05 · Łączenie eye-trackingu i EEG: potencjały wywołane fiksacją (FRP)

**Kurs:** Praktyczna Analiza Danych Eye-Tracking i EEG w Pythonie

W klasycznych badaniach ERP badany patrzy w punkt fiksacji, a bodźce pojawiają się na ekranie. W życiu codziennym jest odwrotnie: to **my przenosimy wzrok** na kolejne obiekty, kilka razy na sekundę. Jednoczesna rejestracja ruchów oczu i EEG (*co-registration*) pozwala badać mózg w takich naturalnych warunkach – zamiast do pojawienia się bodźca odnosimy EEG do **początku fiksacji**. Uśrednione odpowiedzi to **potencjały wywołane fiksacją** (*Fixation-Related Potentials*, FRP; Dimigen i in., 2011).

W tym notebooku:

1. **zsynchronizujemy** zegary eye-trackera i wzmacniacza EEG na podstawie wspólnych znaczników (triggerów),
2. przeniesiemy fiksacje i mrugnięcia z eye-trackera do zapisu EEG,
3. sklasyfikujemy fiksacje jako fiksacje na **celu** i na **dystraktorach** (AOI),
4. usuniemy artefakty oczne (ICA) – przy swobodnym ruchu oczu to krok obowiązkowy,
5. policzymy **FRP** i porównamy fiksacje na celach i dystraktorach (wykresy, mapy, testy statystyczne).

**Dane (symulowane):** 80 prób **wyszukiwania wzrokowego**. Na każdym ekranie było 10 elementów: 1 cel (litera „T”) i 9 dystraktorów („L”). Badany swobodnie przeszukiwał ekran wzrokiem, a po znalezieniu celu naciskał przycisk. Jednocześnie rejestrowano EEG (32 elektrody w układzie BioSemi, 250 Hz, kanały HEOG/VEOG) i ruchy oczu (eye-tracker z zegarem w milisekundach). Podobne badania przeprowadzili m.in. Kamienkowski i in. (2012) oraz Brouwer i in. (2013).

Ponieważ dane są symulowane, znamy prawdę: każda fiksacja wywołuje nad korą potyliczną **odpowiedź lambda** (dodatni załamek ok. 100 ms po początku fiksacji), a pierwsza fiksacja na celu – dodatkowo **P300** nad korą ciemieniową. Zegar eye-trackera „spieszy się” względem zegara EEG o 50 ppm.

**Szacowany czas:** ok. 90–120 minut.

## 0. Przygotowanie środowiska

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from mne.preprocessing import ICA
from mne.stats import permutation_cluster_test
from scipy import stats

KATALOG_GLOWNY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DANE = KATALOG_GLOWNY / "dane"
WYNIKI = KATALOG_GLOWNY / "wyniki"
WYNIKI.mkdir(exist_ok=True)

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")
KOLORY = {"cel": "#eb6834", "dystraktor": "#2a78d6", "tło": "#1baf7a"}  # paleta czytelna przy zaburzeniach widzenia barw
plt.rcParams.update({"figure.figsize": (12, 4)})

# Geometria stanowiska – taka sama jak w notebookach 01–02
EKRAN_SZER_PX, EKRAN_WYS_PX = 1920, 1080
PX_NA_STOPIEN = EKRAN_SZER_PX / 53.1 * 60.0 * np.tan(np.radians(1))  # ≈ 37,9 px

## 1. Wczytanie danych

W prawdziwym badaniu dostajemy osobne pliki z dwóch urządzeń, z **dwoma niezależnymi zegarami**:

| plik | urządzenie | czas |
|---|---|---|
| `eeg_wyszukiwanie_raw.fif` | wzmacniacz EEG; kanał `STI` zawiera triggery | numer próbki EEG (250 Hz) |
| `et_wyszukiwanie_triggery.csv` | te same triggery zapisane przez eye-tracker | zegar eye-trackera [ms] |
| `et_wyszukiwanie_zdarzenia.csv` | fiksacje i mrugnięcia wykryte przez eye-tracker | zegar eye-trackera [ms] |
| `et_wyszukiwanie_bodzce.csv` | położenie elementów na każdym ekranie | – |

Triggery wysyłał program prezentujący bodźce: `11` – początek próby (krzyżyk fiksacyjny), `12` – pojawienie się ekranu wyszukiwania, `13` – reakcja (naciśnięcie przycisku).

In [ ]:
raw = mne.io.read_raw_fif(DANE / "eeg_wyszukiwanie_raw.fif", preload=True)
et_triggery = pd.read_csv(DANE / "et_wyszukiwanie_triggery.csv")
et_zdarzenia = pd.read_csv(DANE / "et_wyszukiwanie_zdarzenia.csv")
bodzce = pd.read_csv(DANE / "et_wyszukiwanie_bodzce.csv")

print(raw)
print(f"Częstotliwość próbkowania EEG: {raw.info['sfreq']:.0f} Hz, kanały EEG: {len(raw.copy().pick('eeg').ch_names)}")
print("Zdarzenia eye-trackera:", et_zdarzenia["typ"].value_counts().to_dict())
et_zdarzenia.head()

In [ ]:
et_triggery.head()

Spójrzmy na jedną próbę: elementy na ekranie i fiksacje wykonane między pojawieniem się ekranu (trigger `12`) a reakcją (trigger `13`). Do tego nie potrzebujemy jeszcze EEG – wszystkie czasy pochodzą z zegara eye-trackera.

In [ ]:
PROBA_PRZYKLAD = 14  # ✏️

t_ekran = et_triggery.query("kod == 12")["czas_et_ms"].to_numpy()[PROBA_PRZYKLAD - 1]
t_reakcja = et_triggery.query("kod == 13")["czas_et_ms"].to_numpy()[PROBA_PRZYKLAD - 1]
fiks_proby = et_zdarzenia.query("typ == 'fiksacja' and proba == @PROBA_PRZYKLAD")
fiks_proby = fiks_proby[(fiks_proby["start_et_ms"] >= t_ekran) & (fiks_proby["start_et_ms"] < t_reakcja)]
elementy = bodzce.query("proba == @PROBA_PRZYKLAD")

fig, ax = plt.subplots(figsize=(11, 6.2))
ax.set_facecolor("#f2f2f2")
for _, el in elementy.iterrows():
    ax.text(el["x_px"], el["y_px"], "T" if el["typ"] == "cel" else "L", fontsize=26, fontweight="bold",
            ha="center", va="center", color=KOLORY["cel"] if el["typ"] == "cel" else "#555555")
ax.plot(fiks_proby["x_px"], fiks_proby["y_px"], color="black", lw=1)
ax.scatter(fiks_proby["x_px"], fiks_proby["y_px"], s=fiks_proby["czas_trwania_ms"] * 2, facecolor="none",
           edgecolor="black", lw=1.5)
for i, (_, f) in enumerate(fiks_proby.iterrows(), start=1):
    ax.annotate(str(i), (f["x_px"], f["y_px"]), xytext=(10, 10), textcoords="offset points", fontsize=11)
ax.set(xlim=(0, EKRAN_SZER_PX), ylim=(EKRAN_WYS_PX, 0), xticks=[], yticks=[],
       title=f"Próba {PROBA_PRZYKLAD}: cel (T), dystraktory (L) i kolejne fiksacje (koła ∝ czas trwania)")
plt.show()

## 2. Synchronizacja zegarów eye-trackera i EEG

Eye-tracker i wzmacniacz EEG mają własne zegary, które:

* startują w różnych chwilach (**przesunięcie**, *offset*),
* chodzą w nieco innym tempie (**dryf**, *drift*) – typowo od kilku do kilkudziesięciu części na milion (ppm), co w ciągu kilkunastu minut nagrania daje od kilku do kilkudziesięciu milisekund błędu.

Program prezentujący bodźce wysyła **te same triggery** do obu urządzeń. Znając czasy wspólnych triggerów w obu zegarach, możemy dopasować prostą:

$$\text{próbka EEG} = a \cdot t_{\text{ET}} + b,$$

gdzie $b$ koryguje przesunięcie, a $a$ – różnicę tempa zegarów. Tak synchronizuje dane np. popularny dodatek EYE-EEG do EEGLAB (Dimigen i in., 2011).

### 2.1. Triggery w EEG i w eye-trackerze

In [ ]:
eeg_triggery = mne.find_events(raw, stim_channel="STI")
print("Kody w EEG:        ", dict(zip(*np.unique(eeg_triggery[:, 2], return_counts=True))))
print("Kody w eye-trackerze:", et_triggery["kod"].value_counts().sort_index().to_dict())

W EEG jest jeden dodatkowy trigger (`99`) – testowy, wysłany zanim eye-tracker zaczął nagrywać. Zostawiamy tylko kody wspólne dla obu urządzeń i sprawdzamy, czy **sekwencje kodów są identyczne**. To bardzo ważny test: jeśli jakiś trigger zginął w jednym z urządzeń, dopasowanie kolejnych par byłoby błędne.

In [ ]:
WSPOLNE_KODY = [11, 12, 13]
eeg_wspolne = eeg_triggery[np.isin(eeg_triggery[:, 2], WSPOLNE_KODY)]

assert len(eeg_wspolne) == len(et_triggery), "Różna liczba triggerów w EEG i w eye-trackerze!"
assert np.array_equal(eeg_wspolne[:, 2], et_triggery["kod"].to_numpy()), "Sekwencje kodów się różnią!"
print(f"OK: {len(eeg_wspolne)} wspólnych triggerów, sekwencje kodów są identyczne.")

probki_eeg = eeg_wspolne[:, 0]  # numer próbki EEG
czasy_et = et_triggery["czas_et_ms"].to_numpy()  # zegar eye-trackera [ms]
FS = raw.info["sfreq"]

### 2.2. Synchronizacja naiwna: tylko przesunięcie

Najprostszy pomysł: wyrównać zegary według **pierwszego** wspólnego triggera i przeliczyć milisekundy na próbki. Sprawdźmy, jak duży błąd popełniamy dla kolejnych triggerów.

In [ ]:
przewidywane_naiwnie = probki_eeg[0] + (czasy_et - czasy_et[0]) * FS / 1000
blad_naiwny_ms = (probki_eeg - przewidywane_naiwnie) / FS * 1000
czas_nagrania_s = (probki_eeg - raw.first_samp) / FS

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(czas_nagrania_s, blad_naiwny_ms, ".", color=KOLORY["dystraktor"])
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="Czas nagrania EEG [s]", ylabel="Błąd [ms]",
       title="Synchronizacja naiwna (tylko przesunięcie): błąd rośnie z czasem")
ax.grid(alpha=0.3)
plt.show()
print(f"Błąd pod koniec nagrania: {blad_naiwny_ms[-1]:.1f} ms")

Błąd rośnie liniowo – to właśnie **dryf zegarów**. Pod koniec nagrania sięga kilkunastu milisekund, a odpowiedź lambda trwa zaledwie kilkadziesiąt. Uśrednianie odpowiedzi przesuniętych o różny czas „rozmazuje” FRP.

### 2.3. Synchronizacja przez regresję liniową

In [ ]:
a, b = np.polyfit(czasy_et, probki_eeg, deg=1)
reszty_ms = (probki_eeg - (a * czasy_et + b)) / FS * 1000
dryf_ppm = (FS / 1000 / a - 1) * 1e6

print(f"Nachylenie a = {a:.8f} próbki EEG / ms zegara ET (nominalnie {FS / 1000:.3f})")
print(f"Oszacowany dryf zegarów: {dryf_ppm:.1f} ppm (zegar eye-trackera spieszy się)")
print(f"Reszty dopasowania: średnio |{np.abs(reszty_ms).mean():.2f}| ms, maksymalnie {np.abs(reszty_ms).max():.2f} ms")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(czas_nagrania_s, reszty_ms, ".", color=KOLORY["dystraktor"])
ax.axhline(0, color="black", lw=0.8)
for granica in (-500 / FS, 500 / FS):
    ax.axhline(granica, color="gray", ls="--", lw=1)
ax.set(xlabel="Czas nagrania EEG [s]", ylabel="Reszta [ms]",
       title="Reszty po synchronizacji liniowej (przerywane linie: ± pół próbki EEG)")
ax.grid(alpha=0.3)
plt.show()


def et_na_probke_eeg(czas_et_ms):
    """Przelicza czas eye-trackera [ms] na numer próbki EEG."""
    return np.round(a * np.asarray(czas_et_ms) + b).astype(int)

Po synchronizacji liniowej reszty nie przekraczają ok. jednej próbki EEG (4 ms) i nie mają żadnego trendu – zostały tylko błędy zaokrąglenia wynikające z częstotliwości próbkowania. Oszacowany dryf jest bliski prawdziwej wartości z symulacji (50 ppm).

> 💡 MNE ma też gotową funkcję `mne.preprocessing.realign_raw`, która wyrównuje dwa nagrania (np. EEG i eye-tracker wczytany jako `Raw`) na podstawie wspólnych zdarzeń.

## 3. Przeniesienie fiksacji i mrugnięć do zapisu EEG

Przeliczamy początki fiksacji i mrugnięć na próbki EEG. Mrugnięcia dodajemy jako **adnotacje** z opisem zaczynającym się od `BAD_` – MNE automatycznie pominie takie fragmenty przy epokowaniu i dopasowaniu ICA. To jedna z zalet jednoczesnej rejestracji: eye-tracker precyzyjnie mówi, kiedy badany mrugał.

In [ ]:
fiksacje = et_zdarzenia.query("typ == 'fiksacja'").reset_index(drop=True)
mrugniecia = et_zdarzenia.query("typ == 'mrugniecie'").reset_index(drop=True)

fiksacje["probka_eeg"] = et_na_probke_eeg(fiksacje["start_et_ms"])
fiksacje["czas_eeg_s"] = (fiksacje["probka_eeg"] - raw.first_samp) / FS

start_mrugniecia_s = (et_na_probke_eeg(mrugniecia["start_et_ms"]) - raw.first_samp) / FS
MARGINES_S = 0.1  # margines wokół mrugnięcia
raw.set_annotations(mne.Annotations(onset=start_mrugniecia_s - MARGINES_S,
                                    duration=mrugniecia["czas_trwania_ms"] / 1000 + 2 * MARGINES_S,
                                    description="BAD_mrugniecie", orig_time=raw.annotations.orig_time))
print(f"Przeniesiono {len(fiksacje)} fiksacji i {len(mrugniecia)} mrugnięć.")

### Kontrola synchronizacji „na oko”

Gdy oko obraca się w poziomie, dipol rogówkowo-siatkówkowy zmienia potencjał w kanale **HEOG** (elektrody przy zewnętrznych kącikach oczu). Jeśli synchronizacja jest poprawna, skoki w HEOG muszą pokrywać się ze zmianami położenia wzroku zarejestrowanymi przez eye-tracker.

In [ ]:
okno_s = (60, 70)  # ✏️ fragment nagrania [s]
w_oknie = fiksacje[(fiksacje["czas_eeg_s"] > okno_s[0]) & (fiksacje["czas_eeg_s"] < okno_s[1])]
heog, czasy = raw.get_data(picks="HEOG", tmin=okno_s[0], tmax=okno_s[1], return_times=True)

fig, osie = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for _, f in w_oknie.iterrows():
    x_deg = (f["x_px"] - EKRAN_SZER_PX / 2) / PX_NA_STOPIEN
    osie[0].hlines(x_deg, f["czas_eeg_s"], f["czas_eeg_s"] + f["czas_trwania_ms"] / 1000,
                   color=KOLORY["dystraktor"], lw=3)
osie[0].set(ylabel="Położenie poziome [°]", title="Eye-tracker: położenie wzroku podczas fiksacji")
osie[1].plot(czasy, heog[0] * 1e6, color="black", lw=0.8)
osie[1].set(xlabel="Czas nagrania EEG [s]", ylabel="HEOG [µV]", title="EEG: kanał HEOG")
for os_ in osie:
    os_.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Klasyfikacja fiksacji: cel czy dystraktor?

Interesują nas fiksacje wykonane, gdy na ekranie były elementy – między triggerami `12` (pojawienie się ekranu) i `13` (reakcja). Każdą taką fiksację przypisujemy do **najbliższego elementu**, jeśli leży bliżej niż 2° od jego środka (to nasze AOI). Fiksacje dalej od wszystkich elementów to fiksacje na tle.

In [ ]:
PROMIEN_AOI_PX = 2 * PX_NA_STOPIEN  # ✏️ promień AOI wokół elementu (2°)

t_ekranow = et_triggery.query("kod == 12")["czas_et_ms"].to_numpy()
t_reakcji = et_triggery.query("kod == 13")["czas_et_ms"].to_numpy()
indeks_proby = fiksacje["proba"].to_numpy() - 1
fiksacje["na_ekranie"] = ((fiksacje["start_et_ms"] >= t_ekranow[indeks_proby])
                          & (fiksacje["start_et_ms"] < t_reakcji[indeks_proby]))

# Każda fiksacja × każdy element tej samej próby → odległość → najbliższy element
pary = fiksacje.reset_index().merge(bodzce, on="proba", suffixes=("", "_el"))
pary["odleglosc"] = np.hypot(pary["x_px"] - pary["x_px_el"], pary["y_px"] - pary["y_px_el"])
najblizsze = pary.loc[pary.groupby("index")["odleglosc"].idxmin()].set_index("index")

fiksacje["obiekt"] = np.where(najblizsze["odleglosc"] < PROMIEN_AOI_PX, najblizsze["typ_el"], "tło")
fiksacje.loc[~fiksacje["na_ekranie"], "obiekt"] = "poza ekranem"
pd.crosstab(fiksacje["obiekt"], fiksacje["na_ekranie"])

Po znalezieniu celu badany często **wraca** na niego wzrokiem. Proces rozpoznania celu (i P300) dotyczy jednak tylko **pierwszej** fiksacji na celu w danej próbie – kolejne fiksacje na tym samym obiekcie odzwierciedlają już inne procesy (np. weryfikację). Dlatego do porównań wybieramy pierwszą fiksację na celu.

In [ ]:
fiksacje["pierwsza_na_celu"] = (fiksacje["obiekt"] == "cel") & ~fiksacje.duplicated(["proba", "obiekt"])

KODY_FIKSACJI = {"fiksacja/cel": 21, "fiksacja/dystraktor": 22, "fiksacja/tło": 23, "fiksacja/inna": 24}
fiksacje["kod"] = np.select(
    [fiksacje["pierwsza_na_celu"], fiksacje["obiekt"] == "dystraktor", fiksacje["obiekt"] == "tło"],
    [21, 22, 23], default=24)
zdarzenia_fiksacji = np.column_stack([fiksacje["probka_eeg"], np.zeros(len(fiksacje), dtype=int), fiksacje["kod"]])

for nazwa, kod in KODY_FIKSACJI.items():
    print(f"{nazwa:20s}: {np.sum(fiksacje['kod'] == kod):4d}")

Fiksacje na celu trwają wyraźnie **dłużej** niż na dystraktorach – badany „zatrzymuje się”, aby potwierdzić rozpoznanie. Zapamiętajmy to: wrócimy do tego w części o pułapkach.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
do_wykresu = fiksacje[fiksacje["kod"].isin([21, 22, 23])].replace({"obiekt": {"cel": "cel (1. fiksacja)"}})
sns.boxplot(data=do_wykresu, x="obiekt", y="czas_trwania_ms", hue="obiekt",
            order=["cel (1. fiksacja)", "dystraktor", "tło"],
            palette={"cel (1. fiksacja)": KOLORY["cel"], "dystraktor": KOLORY["dystraktor"], "tło": KOLORY["tło"]},
            ax=ax)
ax.set(xlabel="", ylabel="Czas fiksacji [ms]", title="Czas trwania fiksacji")
ax.grid(alpha=0.3, axis="y")
plt.show()

Tak wygląda zapis EEG z zaznaczonymi początkami fiksacji i mrugnięciami. Kolor pionowej linii oznacza kategorię fiksacji: **pomarańczowy** – pierwsza fiksacja na celu (kod 21), **niebieski** – dystraktor (22), **zielony** – tło (23), **szary** – pozostałe (24). Czerwone pola to mrugnięcia (`BAD_mrugniecie`). Widać, jak sygnał w kanałach czołowych „podąża” za ruchami oczu.

In [ ]:
kanaly_podgladu = ["Fp1", "Fp2", "F7", "F8", "Fz", "Cz", "Pz", "O1", "Oz", "O2", "HEOG", "VEOG"]
kolory_zdarzen = {21: KOLORY["cel"], 22: KOLORY["dystraktor"], 23: KOLORY["tło"], 24: "gray"}
fig = raw.copy().pick(kanaly_podgladu).filter(0.1, 30, picks=["eeg", "eog"]).plot(
    events=zdarzenia_fiksacji, event_color=kolory_zdarzen, start=60, duration=8,
    scalings=dict(eeg=50e-6, eog=300e-6), show_scrollbars=False, show=False)
fig.suptitle("EEG z zaznaczonymi początkami fiksacji i mrugnięciami", y=1.01, fontweight="bold")
plt.show()

## 5. Przygotowanie EEG

### 5.1. Filtracja i referencja

Dane nagrano w Polsce – w widmie widać pik **50 Hz** od sieci energetycznej. Filtr pasmowy 0,1–30 Hz (typowy w analizach ERP/FRP) usunie go razem z wolnymi dryfami. Następnie stosujemy referencję do średniej.

In [ ]:
fig = raw.compute_psd(fmax=80, picks="eeg", reject_by_annotation=False).plot(amplitude=False, show=False)
fig.axes[0].axvline(50, color="gray", ls="--", lw=1)
fig.axes[0].set_title("Widmo mocy surowego EEG – pik 50 Hz")
plt.show()

raw_filtr = raw.copy().filter(l_freq=0.1, h_freq=30.0, picks=["eeg", "eog"])
raw_filtr.set_eeg_reference("average", projection=False)

### 5.2. Dlaczego bez usunięcia artefaktów ocznych się nie obejdzie?

Przy swobodnym oglądaniu oczy wykonują 3–4 sakady na sekundę, a każda z nich zmienia potencjał w elektrodach (najsilniej w czołowych). Spróbujmy wyciąć epoki wokół fiksacji z typowym progiem odrzucania 100 µV:

In [ ]:
epoki_bez_ica = mne.Epochs(raw_filtr, zdarzenia_fiksacji, event_id={"fiksacja/cel": 21, "fiksacja/dystraktor": 22},
                           tmin=-0.2, tmax=0.6, baseline=(-0.2, 0), reject=dict(eeg=100e-6), preload=True)
print(f"Bez usunięcia artefaktów ocznych odrzucono {epoki_bez_ica.drop_log_stats():.0f}% epok!")

Odrzucilibyśmy większość danych. Zamiast odrzucać epoki, **usuniemy artefakty oczne za pomocą ICA** (jak w notebooku 03). W badaniach ze swobodnym ruchem oczu warto sięgnąć po procedurę OPTICAT (Dimigen, 2020), która dodatkowo „uczy” ICA na fragmentach wokół sakad.

Mamy tu tylko 32 elektrody, więc wyznaczamy 15 składowych. Przy tak małej liczbie składowych domyślne kryterium `find_bads_eog` (wartość *z* > 3) bywa zawodne – dwie silne składowe oczne „zawyżają” odchylenie standardowe wyników. Dlatego składowe oczne wybieramy prostszym kryterium: **bezwzględna korelacja z kanałem EOG większa niż 0,5**.

In [ ]:
raw_do_ica = raw.copy().filter(l_freq=1.0, h_freq=30.0, picks=["eeg", "eog"])
raw_do_ica.set_eeg_reference("average", projection=False)

ica = ICA(n_components=15, method="fastica", random_state=97, max_iter="auto")
ica.fit(raw_do_ica, picks="eeg")  # fragmenty BAD_mrugniecie są automatycznie pomijane

eog_idx, eog_wyniki = ica.find_bads_eog(raw_do_ica, ch_name=["HEOG", "VEOG"], measure="correlation", threshold=0.5)
eog_idx = [int(i) for i in eog_idx]
print("Składowe oczne:", eog_idx)
fig = ica.plot_scores(eog_wyniki, exclude=eog_idx, title="Korelacja składowych ICA z HEOG (góra) i VEOG (dół)")

In [ ]:
fig = ica.plot_components()

In [ ]:
ica.exclude = eog_idx
raw_czyste = ica.apply(raw_filtr.copy())

## 6. Potencjały wywołane fiksacją (FRP)

Epoki wycinamy tak jak w notebooku 04, ale punktem odniesienia (czas 0) jest **początek fiksacji**. Odrzucamy epoki nakładające się na mrugnięcia (`reject_by_annotation=True`) i epoki z pozostałymi dużymi artefaktami.

In [ ]:
frp = mne.Epochs(raw_czyste, zdarzenia_fiksacji, event_id={"fiksacja/cel": 21, "fiksacja/dystraktor": 22},
                 tmin=-0.2, tmax=0.6, baseline=(-0.2, 0), reject=dict(eeg=100e-6),
                 reject_by_annotation=True, picks=["eeg"], preload=True)
print(frp)
print(f"Po ICA odrzucono tylko {frp.drop_log_stats():.1f}% epok")

frp_cel = frp["fiksacja/cel"].average()
frp_dystraktor = frp["fiksacja/dystraktor"].average()

### 6.1. Odpowiedź lambda – wspólna dla wszystkich fiksacji

Uśredniamy wszystkie fiksacje razem. Ok. 100 ms po początku fiksacji nad korą potyliczną pojawia się dodatni załamek – **odpowiedź lambda**. To wzrokowa odpowiedź na nowy obraz, który „spada” na siatkówkę po każdej sakadzie (odpowiednik P1 z klasycznych ERP).

In [ ]:
fig = frp.average().plot_joint(times=[0.1, 0.17, 0.4], title="FRP – wszystkie fiksacje (cel + dystraktory)")

### 6.2. Cel vs dystraktor

Porównujemy FRP dla pierwszej fiksacji na celu i dla fiksacji na dystraktorach w dwóch kanałach: **Oz** (odpowiedź lambda) i **Pz** (P300). Pas wokół krzywej to ± SEM.

In [ ]:
def pasmo_sem(dane):
    """Granice pasma średnia ± SEM; dane: tablica (obserwacje × czas)."""
    srednia = dane.mean(axis=0)
    sem = dane.std(axis=0, ddof=1) / np.sqrt(len(dane))
    return np.array([srednia - sem, srednia + sem])


proby = {"cel": list(frp["fiksacja/cel"].iter_evoked()),
         "dystraktor": list(frp["fiksacja/dystraktor"].iter_evoked())}
fig, osie = plt.subplots(1, 2, figsize=(14, 4.5))
for os_, kanal in zip(osie, ["Oz", "Pz"]):
    mne.viz.plot_compare_evokeds(proby, picks=kanal, ci=pasmo_sem, axes=os_, show=False,
                                 colors={"cel": KOLORY["cel"], "dystraktor": KOLORY["dystraktor"]},
                                 title=f"{kanal}: FRP (średnia ± SEM)", show_sensors="upper right")
plt.show()

* W **Oz** odpowiedź lambda jest praktycznie taka sama dla celów i dystraktorów – to wczesna, sensoryczna reakcja na każdą fiksację.
* W **Pz** po fiksacji na celu pojawia się szeroki dodatni komponent ok. 300–500 ms – **P300**, związany z rozpoznaniem poszukiwanego obiektu. Fiksacje na dystraktorach go nie wywołują.

### 6.3. Mapy topograficzne

Mapy potwierdzają różną lokalizację obu komponentów: odpowiedź lambda – potylicznie, P300 – ciemieniowo.

In [ ]:
roznica = mne.combine_evoked([frp_cel, frp_dystraktor], weights=[1, -1])
fig, osie = plt.subplots(1, 3, figsize=(12, 4))
for os_, (evoked, t0, t1, opis) in zip(osie, [(frp_dystraktor, 0.08, 0.12, "Dystraktor: lambda (80–120 ms)"),
                                             (frp_cel, 0.3, 0.5, "Cel: 300–500 ms"),
                                             (roznica, 0.3, 0.5, "Cel − dystraktor: 300–500 ms")]):
    wartosci = evoked.copy().crop(t0, t1).data.mean(axis=1) * 1e6
    obraz_mapy, _ = mne.viz.plot_topomap(wartosci, evoked.info, axes=os_, vlim=(-6, 6), cmap="RdBu_r", show=False)
    os_.set_title(opis)
fig.colorbar(obraz_mapy, ax=osie, label="Średnia amplituda [µV]", shrink=0.7)
plt.show()

### 6.4. Test statystyczny

**Podejście 1: średnia amplituda w oknie.** Dla każdej epoki liczymy średnią amplitudę w Pz w oknie 300–500 ms (okno wybrane *a priori* na podstawie literatury o P300) i porównujemy grupy epok testem t Welcha.

In [ ]:
OKNO_P300 = (0.3, 0.5)
amplitudy = {nazwa: frp[f"fiksacja/{nazwa}"].copy().crop(*OKNO_P300).get_data(picks="Pz").mean(axis=(1, 2)) * 1e6
             for nazwa in ["cel", "dystraktor"]}
t, p = stats.ttest_ind(amplitudy["cel"], amplitudy["dystraktor"], equal_var=False)
d = (amplitudy["cel"].mean() - amplitudy["dystraktor"].mean()) / np.sqrt(
    (amplitudy["cel"].var(ddof=1) + amplitudy["dystraktor"].var(ddof=1)) / 2)
for nazwa, wartosci in amplitudy.items():
    print(f"{nazwa:10s}: {wartosci.mean():5.2f} ± {stats.sem(wartosci):.2f} µV (n = {len(wartosci)})")
print(f"Test t Welcha: t = {t:.2f}, p = {p:.2g}, d Cohena = {d:.2f}")

**Podejście 2: test permutacyjny oparty na klastrach** (Maris i Oostenveld, 2007). Zamiast wybierać okno z góry, testujemy każdy punkt czasowy, a problem wielu porównań rozwiązujemy, oceniając istotność całych **klastrów** sąsiadujących punktów za pomocą permutacji.

In [ ]:
X = [frp[f"fiksacja/{nazwa}"].get_data(picks="Pz")[:, 0, :] * 1e6 for nazwa in ["cel", "dystraktor"]]
F_obs, klastry, p_klastrow, _ = permutation_cluster_test(X, n_permutations=1000, tail=1, seed=42,
                                                         out_type="mask", verbose=False)

czasy_ms = frp.times * 1000
fig, ax = plt.subplots(figsize=(11, 4))
for nazwa, dane in zip(["cel", "dystraktor"], X):
    srednia, sem = dane.mean(axis=0), stats.sem(dane, axis=0)
    ax.plot(czasy_ms, srednia, color=KOLORY[nazwa], lw=2, label=nazwa)
    ax.fill_between(czasy_ms, srednia - sem, srednia + sem, color=KOLORY[nazwa], alpha=0.25, lw=0)
for klaster, p_k in zip(klastry, p_klastrow):
    if p_k < 0.05:
        ax.axvspan(czasy_ms[klaster][0], czasy_ms[klaster][-1], color="gray", alpha=0.2)
        print(f"Istotny klaster: {czasy_ms[klaster][0]:.0f}–{czasy_ms[klaster][-1]:.0f} ms, p = {p_k:.3f}")
ax.axvline(0, color="black", lw=1)
ax.axhline(0, color="black", lw=0.5)
ax.set(xlabel="Czas od początku fiksacji [ms]", ylabel="Amplituda [µV]",
       title="Pz: cel vs dystraktor (szare tło – istotne klastry, p < 0,05)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

Uwaga: w tej analizie obserwacjami są **epoki jednej osoby**. W prawdziwym badaniu wnioskujemy o populacji, więc testujemy średnie FRP **wielu uczestników** (np. test t dla prób zależnych lub test klastrowy na danych grupowych).

## 7. Pułapki analizy FRP

Analiza FRP ma kilka specyficznych problemów (Dimigen i in., 2011; Nikolaev i in., 2016):

1. **Nakładanie się odpowiedzi.** Kolejna fiksacja zaczyna się zwykle po 200–300 ms, więc w epoce od −200 do 600 ms „siedzą” też odpowiedzi na fiksację poprzednią i następną.
2. **Różny czas fiksacji w warunkach.** Fiksacje na celu trwały dłużej (patrz wykres w sekcji 4), więc w ich epokach odpowiedź na **następną** fiksację pojawia się później niż w epokach dystraktorów. Część różnicy między warunkami może wynikać z samego nakładania się odpowiedzi, a nie z procesów poznawczych.
3. **Właściwości sakad.** Amplituda odpowiedzi lambda zależy od amplitudy poprzedzającej sakady. Jeśli sakady na cele i dystraktory różnią się wielkością, różnią się też FRP.
4. **Linia bazowa.** Okres przed fiksacją zawiera sakadę (i np. odpowiedź na pojawienie się ekranu), więc linia bazowa nie jest „czysta”.

Rozwiązaniem jest **dekonwolucja** i modelowanie regresyjne: zamiast uśredniać, modelujemy sygnał jako sumę nakładających się odpowiedzi na wszystkie zdarzenia, z uwzględnieniem kowariantów, np. amplitudy sakady (pakiet *Unfold*; Ehinger i Dimigen, 2019). W MNE prostą dekonwolucję oferuje funkcja `mne.stats.linear_regression_raw`.

## 8. Zapis wyników

In [ ]:
mne.write_evokeds(WYNIKI / "05_frp-ave.fif", [frp_cel, frp_dystraktor], overwrite=True)
fiksacje.to_csv(WYNIKI / "05_fiksacje_zsynchronizowane.csv", index=False)
print("Zapisano:", *sorted(p.name for p in WYNIKI.glob("05_*")), sep="\n  ")

## Podsumowanie

* Dane z eye-trackera i EEG łączymy przez **wspólne triggery**; regresja liniowa koryguje zarówno przesunięcie, jak i dryf zegarów. Zawsze sprawdzamy zgodność sekwencji triggerów i reszty dopasowania.
* Eye-tracker dostarcza EEG cennych informacji: **kiedy** zaczęła się fiksacja, **gdzie** padł wzrok (AOI) i **kiedy** badany mrugał.
* Przy swobodnym ruchu oczu artefakty oczne trzeba usuwać (ICA), a nie odrzucać epoki.
* FRP pokazują wczesną odpowiedź **lambda** na każdą fiksację oraz późniejsze komponenty zależne od tego, *na co* patrzymy – tu **P300** po znalezieniu celu.
* Nakładanie się odpowiedzi i różnice w ruchach oczu między warunkami to główne pułapki interpretacji FRP.

## Zadania do samodzielnego wykonania

Pięć zadań, w których powtórzysz najważniejsze kroki z tego notebooka. **Nie wymagają nowych funkcji ani pojęć** – wszystko, czego potrzebujesz, pojawiło się wyżej (przy każdym zadaniu podajemy, którą sekcję powtarza). Każde zadanie kończy się **jednym wynikiem**: liczbą wypisaną w podanym formacie albo jedną figurą.

**Zasady pracy**

1. Zanim zaczniesz, uruchom cały notebook (**Run All**) – zadania korzystają ze zmiennych i funkcji zdefiniowanych wyżej.
2. Rozwiązanie wpisz w komórce kodu pod treścią zadania.
3. **Nie nadpisuj zmiennych z notebooka** (np. `raw`, `fiksacje`, `frp`). Nowym zmiennym nadawaj nazwy z przedrostkiem zadania: `z1_`, `z2_`, … Dzięki temu zadania nie psują sobie nawzajem danych i nie trzeba uruchamiać notebooka od nowa.
4. **Kontrola pośrednia** to wartość, którą możesz sprawdzić po drodze. Jeśli się nie zgadza, błąd jest wcześniej – popraw go, zanim pójdziesz dalej.
5. **✅ Wynik** to jedyna rzecz do oddania: liczba wypisana dokładnie w podanym formacie (z podaną liczbą miejsc po przecinku) albo figura z podanym tytułem.
6. **Do przemyślenia** – pytanie, które pomaga zrozumieć wynik. Nie trzeba go oddawać, ale warto na nie odpowiedzieć choćby w myślach.

### Zadanie 1. Najdłuższe wyszukiwanie

*Powtarzasz: sekcje 1 i 4 (triggery eye-trackera, rysowanie pojedynczej próby).*

**Cel:** przećwiczyć pracę z triggerami i fiksacjami w zegarze eye-trackera.

**Krok po kroku:**

1. Czas wyszukiwania w każdej próbie to różnica między triggerem reakcji (`13`) a triggerem pojawienia się ekranu (`12`) w zegarze eye-trackera. Oba czasy masz w tablicach `t_reakcji` i `t_ekranow` z sekcji 4 – są uporządkowane według numeru próby.
2. Znajdź próbę z najdłuższym czasem wyszukiwania (`np.argmax`). Pamiętaj, że indeksy tablicy zaczynają się od 0, a numery prób – od 1.
3. Narysuj tę próbę tak jak w sekcji 1: skopiuj kod komórki z `PROBA_PRZYKLAD`, ustaw numer znalezionej próby i zmień nazwy zmiennych na `z1_…`, żeby nie nadpisać tych z notebooka.
4. Tytuł wykresu zacznij od „Zadanie 1” i podaj w nim numer próby, czas wyszukiwania [ms] i liczbę fiksacji wykonanych między triggerami `12` i `13`.

**Kontrola pośrednia:** średni czas wyszukiwania (ze wszystkich prób) to ok. 1773 ms.

**✅ Wynik:** jedna figura z najdłuższą próbą (numer próby, czas wyszukiwania i liczba fiksacji w tytule).

**Do przemyślenia:** Ile dystraktorów badany obejrzał, zanim trafił na cel? Czy na niektóre elementy patrzył więcej niż raz?

In [ ]:
# Zadanie 1 – Twoje rozwiązanie



# Wynik: jedna figura z tytułem zaczynającym się od „Zadanie 1”

### Zadanie 2. Co psuje zła synchronizacja?

*Powtarzasz: sekcje 2 (synchronizacja zegarów), 4 (tablica zdarzeń) i 6 (FRP).*

**Cel:** sprawdzić, jak dryf zegarów – jeśli go nie skorygujemy – zniekształca odpowiedź lambda.

**Krok po kroku:**

1. Policz próbki EEG początków fiksacji metodą **naiwną** z sekcji 2.2 (tylko przesunięcie względem pierwszego wspólnego triggera): zamiast funkcji `et_na_probke_eeg` użyj wzoru `probki_eeg[0] + (czas_et - czasy_et[0]) * FS / 1000`, gdzie `czas_et` to kolumna `fiksacje["start_et_ms"]`. Wynik zaokrąglij do liczb całkowitych: `np.round(...).astype(int)`.
2. Zbuduj tablicę zdarzeń tak jak `zdarzenia_fiksacji` w sekcji 4 (trzy kolumny: próbka, zera, kod), ale z naiwnymi próbkami. Kody zostają te same (`fiksacje["kod"]`).
3. Wytnij epoki z `raw_czyste` dla kodów 21 i 22, z tymi samymi parametrami co `frp` w sekcji 6.
4. Uśrednij wszystkie epoki (cele i dystraktory razem) i policz średnią amplitudę w kanale `Oz` w oknie odpowiedzi lambda **80–120 ms** (µV).
5. Zrób to samo dla poprawnie zsynchronizowanych epok (`frp.average()`) i porównaj.

**Kontrola pośrednia:** przy poprawnej synchronizacji średnia amplituda wynosi ok. 4,7 µV.

**✅ Wynik:** `print(f"Zadanie 2: lambda w Oz przy synchronizacji naiwnej = {z2_naiwna:.1f} µV")`

**Do przemyślenia:** O ile próbek (i milisekund) przesunięte są naiwne zdarzenia na początku, a o ile na końcu nagrania? Dlaczego odpowiedź po uśrednieniu jest niższa, chociaż w żadnej pojedynczej epoce nie zmieniła się jej amplituda? Co by się stało w nagraniu trwającym godzinę?

In [ ]:
# Zadanie 2 – Twoje rozwiązanie



# print(f"Zadanie 2: lambda w Oz przy synchronizacji naiwnej = {z2_naiwna:.1f} µV")

### Zadanie 3. Jak duży powinien być AOI?

*Powtarzasz: sekcja 4 (klasyfikacja fiksacji: cel, dystraktor, tło).*

**Cel:** sprawdzić, jak promień AOI wpływa na klasyfikację fiksacji.

**Krok po kroku:**

1. W sekcji 4 tabela `najblizsze` zawiera dla każdej fiksacji typ najbliższego elementu (`typ_el`) i odległość od niego w pikselach (`odleglosc`). Nie musisz liczyć tego od nowa.
2. Zrób kopię tabeli fiksacji (`z3 = fiksacje.copy()`) i sklasyfikuj w niej fiksacje jeszcze raz, z promieniem AOI **1°** zamiast 2° (`1 * PX_NA_STOPIEN`). Skopiuj linię z `np.where(...)` z sekcji 4 i zmień tylko promień; wynik zapisz w kolumnie `z3["obiekt"]`.
3. Weź pod uwagę tylko fiksacje wykonane, gdy na ekranie były elementy (`z3["na_ekranie"]`), i policz, ile z nich trafiło do każdej kategorii (`value_counts()`).

**Kontrola pośrednia:** przy promieniu 2° (jak w notebooku) jest 80 fiksacji na celu – po jednej w każdej próbie.

**✅ Wynik:** `print(f"Zadanie 3: przy promieniu 1° fiksacji na celu jest {z3_liczby['cel']}")`

**Do przemyślenia:** Typowa dokładność eye-trackera to 0,5–1°. Dlaczego zbyt mały AOI „gubi” prawdziwe fiksacje na celu? Co mogłoby się stać przy bardzo dużym promieniu, gdyby elementy leżały blisko siebie?

In [ ]:
# Zadanie 3 – Twoje rozwiązanie



# print(f"Zadanie 3: przy promieniu 1° fiksacji na celu jest {z3_liczby['cel']}")

### Zadanie 4. Czy odpowiedź lambda potrzebuje obrazu?

*Powtarzasz: sekcje 4 (kody fiksacji) i 6 (epokowanie FRP, odpowiedź lambda).*

**Cel:** przećwiczyć wycinanie FRP dla wybranej kategorii fiksacji i sprawdzić, od czego zależy odpowiedź lambda.

**Krok po kroku:**

1. Fiksacje z kodem `24` („fiksacja/inna”) to fiksacje wykonane, gdy na ekranie **nie było** elementów wyszukiwania (krzyżyk fiksacyjny, pusty ekran między próbami).
2. Wytnij epoki z `raw_czyste` dla kodu `24` i – osobno – dla kodu `22` (dystraktory). Użyj tablicy `zdarzenia_fiksacji` i tych samych parametrów co `frp` w sekcji 6 (`tmin`, `tmax`, `baseline`, `reject`, `reject_by_annotation`, `picks`).
3. Dla każdego kodu uśrednij epoki i policz średnią amplitudę w kanale `Oz` w oknie 80–120 ms (µV). Zapisz też liczbę epok, które zostały po odrzuceniu.

**Kontrola pośrednia:** dla dystraktorów wychodzi ok. 4,5 µV (277 epok).

**✅ Wynik:** `print(f"Zadanie 4: lambda w Oz dla kodu 24 = {z4_amp_24:.1f} µV (n = {z4_n_24})")`

**Do przemyślenia:** Co ten wynik mówi o tym, czym jest odpowiedź lambda (wróć do opisu w sekcji 6.1)? Dlaczego z fiksacji z kodem 24 odrzucono tak wiele epok? (Sprawdź `drop_log` – kiedy badany najczęściej mruga?)

In [ ]:
# Zadanie 4 – Twoje rozwiązanie



# print(f"Zadanie 4: lambda w Oz dla kodu 24 = {z4_amp_24:.1f} µV (n = {z4_n_24})")

### Zadanie 5. Pułapka: amplituda sakady

*Powtarzasz: sekcje 6.4 (średnia amplituda w oknie i test t Welcha) i 7 (pułapki analizy FRP, punkt 3).*

**Cel:** sprawdzić na danych trzecią pułapkę z sekcji 7 – czy odpowiedź lambda zależy od amplitudy sakady, która doprowadziła do fiksacji.

**Krok po kroku:**

1. Zrób kopię tabeli: `z5 = fiksacje.copy()`. Policz amplitudę sakady prowadzącej do każdej fiksacji, czyli odległość od **poprzedniej** fiksacji w stopniach. Fiksacje są ułożone w kolejności czasowej, więc wystarczy przesunięcie o jeden wiersz (`shift()`, jak przy wizytach w notebooku 02):
   `np.hypot(z5["x_px"] - z5["x_px"].shift(), z5["y_px"] - z5["y_px"].shift()) / PX_NA_STOPIEN`
2. Weź fiksacje na dystraktorach (`z5["kod"] == 22`) i podziel je **medianą** amplitudy sakady na „małe” (≤ mediana) i „duże” (> mediana).
3. Utwórz nową kolumnę z kodami: skopiuj `z5["kod"]`, a potem nadaj fiksacjom „małym” kod `31`, a „dużym” – `32` (`z5.loc[maska, "kolumna"] = …`). Zbuduj z niej tablicę zdarzeń jak w sekcji 4 i wytnij epoki z `raw_czyste` z parametrami jak w sekcji 6, z `event_id={"małe": 31, "duże": 32}`.
4. Dla każdej epoki policz średnią amplitudę w `Oz` w oknie 80–120 ms i porównaj grupy testem t Welcha – dokładnie tak jak w sekcji 6.4 (tam: `Pz`, 300–500 ms). Kolejność w teście: najpierw „duże”, potem „małe”.

**Kontrola pośrednia:** mediana amplitudy sakad prowadzących do fiksacji na dystraktorach wynosi ok. 13,3°.

**✅ Wynik:** `print(f"Zadanie 5: duże − małe = {z5_roznica:.1f} µV, t = {z5_t:.2f}, p = {z5_p:.2f}")`

**Do przemyślenia:** W jakim kierunku jest różnica i czy jest istotna statystycznie przy tej liczbie epok? Wyobraź sobie, że sakady prowadzące do celów są średnio większe niż te prowadzące do dystraktorów – jak taka różnica mogłaby „udawać” efekt warunku w porównaniu cel vs dystraktor?

In [ ]:
# Zadanie 5 – Twoje rozwiązanie



# print(f"Zadanie 5: duże − małe = {z5_roznica:.1f} µV, t = {z5_t:.2f}, p = {z5_p:.2f}")

## Literatura

* Brouwer, A.-M., Reuderink, B., Vincent, J., van Gerven, M. A. J., i van Erp, J. B. F. (2013). Distinguishing between target and nontarget fixations in a visual search task using fixation-related potentials. *Journal of Vision, 13*(3), 17.
* Dimigen, O. (2020). Optimizing the ICA-based removal of ocular EEG artifacts from free viewing experiments. *NeuroImage, 207*, 116117.
* Dimigen, O., Sommer, W., Hohlfeld, A., Jacobs, A. M., i Kliegl, R. (2011). Coregistration of eye movements and EEG in natural reading: Analyses and review. *Journal of Experimental Psychology: General, 140*(4), 552–572.
* Ehinger, B. V., i Dimigen, O. (2019). Unfold: An integrated toolbox for overlap correction, non-linear modeling, and regression-based EEG analysis. *PeerJ, 7*, e7838.
* Kamienkowski, J. E., Ison, M. J., Quiroga, R. Q., i Sigman, M. (2012). Fixation-related potentials in visual search: A combined EEG and eye tracking study. *Journal of Vision, 12*(7), 4.
* Maris, E., i Oostenveld, R. (2007). Nonparametric statistical testing of EEG- and MEG-data. *Journal of Neuroscience Methods, 164*(1), 177–190.
* Nikolaev, A. R., Meghanathan, R. N., i van Leeuwen, C. (2016). Combining EEG and eye movement recording in free viewing: Pitfalls and possibilities. *Brain and Cognition, 107*, 55–83.